# CIS515

In [0]:
##### DO NOT EDIT THIS CELL!!!
%pip install --upgrade mlflow tensorflow optuna
dbutils.library.restartPython()

In [0]:
##### DO NOT EDIT THIS CELL!!!
import numpy as np
import mlflow
import mlflow.keras
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import matplotlib.pyplot as plt
import os


**Image Preprocessing**

In [0]:
##### DO NOT EDIT THIS CELL!!!
# # Load Fashion-MNIST 
(X_train_full, y_train_full), (X_test_clean, y_test_clean) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize  images to lie in the range 0-1 and add channel dimension for CNN (28x28x1)
X_train_full = X_train_full / 255.0
X_test_clean = X_test_clean / 255.0
X_train_full = X_train_full[..., np.newaxis]   # (60000, 28, 28, 1)
X_test_clean = X_test_clean[..., np.newaxis]   # (10000, 28, 28, 1)

class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"]
num_classes = len(class_names)

print(f"Train: {X_train_full.shape}  Test: {X_test_clean.shape}")
print(f"Classes: {class_names}")


## #ADD YOUR ANSWER TO PART 1 QUESTION 1 HERE#[](url)



**Data Exploration**

We study data drift and how it affects model performance.

Fashion-MNIST represents clothing images from a controlled studio environment. In production, images arrive from retail store cameras, smartphones, and low-quality scanners. Thus, images may be imperfect due to noise, occlusion, and lighting changes. This simulates real-world deployment drift.

**Sample Images from Fashion-MNIST**

In [0]:
##### DO NOT EDIT THIS CELL!!!
# # Show sample images from each class
plt.figure(figsize=(10, 4))
for i in range(10):
    idx = np.where(y_train_full == i)[0][0]
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train_full[idx].squeeze(), cmap="gray")
    plt.title(class_names[i], fontsize=8)
    plt.axis("off")
plt.suptitle("Fashion-MNIST Classes (Clean Training Data)")
plt.tight_layout()
plt.show()


In [0]:
# Class distribution
unique, counts = np.unique(y_train_full, return_counts=True)
plt.figure()
plt.bar([class_names[i] for i in unique], counts)
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Training Class Distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Drift Simulation & Producing Degraded Images**

In [0]:
# Simulate 3 types of real-world image drift

# TYPE 1: Gaussian Noise — poor camera sensor, low-light grain
noise = np.random.normal(0, 0.25, X_test_clean.shape)
X_test_noisy = np.clip(X_test_clean + noise, 0, 1)

# TYPE 2: Occlusion — objects partially blocked (stickers, tags, shelving)
X_test_occluded = X_test_clean.copy()
for img in X_test_occluded:
    # random black rectangle covering ~25% of image
    r, c = np.random.randint(0, 14), np.random.randint(0, 14)
    img[r:r+14, c:c+14, :] = 0.0

# TYPE 3: Low quality / dark — cheap scanner, poor lighting in store
X_test_dark = np.clip(X_test_clean - 0.35, 0, 1)

# Show side by side comparison
fig, axes = plt.subplots(4, 5, figsize=(12, 9))
titles = ["Clean", "Noisy", "Occluded (Blocked)", "Dark (Poor Light)"]
datasets = [X_test_clean, X_test_noisy, X_test_occluded, X_test_dark]
for row, (data, title) in enumerate(zip(datasets, titles)):
    for col in range(5):
        axes[row, col].imshow(data[col].squeeze(), cmap="gray")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(title, fontsize=9, rotation=90, labelpad=40)
plt.suptitle("Drift Types: Clean vs Production Conditions", fontsize=13)
plt.tight_layout()
plt.show()

print("Drift datasets created:")
print(f"  X_test_noisy    shape: {X_test_noisy.shape}")
print(f"  X_test_occluded shape: {X_test_occluded.shape}")
print(f"  X_test_dark     shape: {X_test_dark.shape}")



## #ADD YOUR ANSWER TO PART 1 QUESTION 3 HERE#[](url)

**Train / Val Split**

In [0]:
from sklearn.model_selection import train_test_split

# Split: 80% train, 20% val — using clean pre-drift training data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}")


**Define Model Architecture**

In [0]:
mlflow.tensorflow.autolog(disable=True)  # log manually inside trial

#EDIT THIS CELL

###################PART 2 BUILD CNN MODEL ###################
def build_keras(conv1_filters, conv2_filters, learning_rate):
# CREATE model 
# a sequential model is appropriate for a plain stack of layers where each layer 
# has exactly one input tensor and one output tensor. 

  ### (1) ADD YOUR CODE HERE
    model = ###
# set the input shape is (28, 28, 1); 
# sample size is ommitted - only the shape of each sample is specified 
       


### (2) YOUR CODE HERE
# Create a pair of convolutional-pooling layers as below (leave everything else at default values)
    #  - Conv2D: 8 filters, kernel size (3,3), strides (1,1), valid padding, relu activation
    #  - MaxPool2D: pool size (2,2), strides (2,2)  
       
### (3) YOUR CODE HERE
    # Create another pair of convolutional-pooling layers as below (leave everything else at default values)
    #  - Conv2D: 16 filters, kernel size (3,3), strides (1,1), valid padding, relu activation
    #  - MaxPool2D: pool size (2,2), strides (2,2)
        
### (4) YOUR CODE HERE
    # Flatten to get output ready for fully connected layer (leave everything at default values)
       
### (5) YOUR CODE HERE
# create dense output layer with 128 nodes and relu activation (leave everything else at default values)
       
### (6) YOUR CODE HERE
        #add a dropout layer 
   
        # create dense output layer with Softmax activation (leave everything else at default values)
   

### (6) YOUR CODE HERE
# use Adam optimizer with learning rate 0.001 (leave everything else at default values)
# use SparseCategoricalCrossentropy loss (leave everything at default values)
 # track accuracy metric
# COMPILE model
   
    return model



**Hyperparameter Search with Optuna + MLflow**


## #ADD YOUR ANSWER TO PART 3 QUESTION 1 HERE#[](url)

In [0]:
import optuna
#Log hyperparamteres
#############PART 3 ################
N_TRIALS = #####

def objective(trial):
    ### (1) YOUR CODE HERE
    #Add hyperparamters value ranges
    conv1_filters  = trial.suggest_categorical("conv1_filters", ##add values here for Q3.1 ###)
    conv2_filters  = trial.suggest_categorical("conv2_filters", ##add values here for Q3.1 ###)
    learning_rate  = trial.suggest_float("learning_rate", ##add values here for Q3.1 ###, log=True)
    epochs         = trial.suggest_int("epochs", ##add values here for Q3.1 ###)
    batch_size     = trial.suggest_categorical("batch_size", ##add values here for Q3.1 ###)

    with mlflow.start_run(nested=True) as run:
         ### (2) YOUR CODE HERE
         #Log trial hyperparamters for each run
        mlflow.log_params(trial.params)

        model = build_keras(conv1_filters, conv2_filters, learning_rate)

        try:
            model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=epochs,
                batch_size=batch_size,
                verbose=0
            )
        except optuna.exceptions.TrialPruned:
            mlflow.set_tag("optuna_pruned", True)
            raise

        _, val_acc = model.evaluate(X_val, y_val, verbose=0)
         ### (3) YOUR CODE HERE
         #log validation accuracy for each run and model
        
        ### (4) Record each run and model url
        #use names:#"mlflow_run_id" #"mlflow_model_uri"
       

    return val_acc
### (5) YOUR CODE HERE
#In optuna module, create study object and define  optimizaiton direction
study = ######  
study.optimize(objective, n_trials=N_TRIALS)


In [0]:
best = study.best_trial
print("Best trial params:", best.params, "val_accuracy:", best.value)
best_run_id   = best.user_attrs.get("mlflow_run_id")
best_model_uri = best.user_attrs.get("mlflow_model_uri")
print("Best model uri:", best_model_uri)


**LOAD SCREENSHOTS FOR PART 4.1 HERE**



**ADD YOUR ANSWER TO PART 4 QUESTION 2 HERE#[](url)**

**Record the Best Model & Create Signature**

In [0]:
import tempfile
from mlflow.models.signature import infer_signature

os.makedirs("/tmp/mlflow_tmp", exist_ok=True)
os.environ["MLFLOW_TMP_DIR"] = "/tmp/mlflow_tmp"
tempfile.tempdir = "/tmp/mlflow_tmp"

# Load best model from Optuna run
best_model = mlflow.keras.load_model(best_model_uri)

# Infer signature from clean validation data
X_sample_np = np.array(X_val[:5], dtype=np.float32)
y_pred_sample = best_model.predict(X_sample_np)
signature = infer_signature(X_sample_np, y_pred_sample)

with mlflow.start_run(run_name="best_model_final") as run:
    mlflow.keras.log_model(
        best_model,
        name="best_model_sign",
        signature=signature,
    )
    best_run_id   = run.info.run_id
    best_model_uri = f"runs:/{run.info.run_id}/best_model_sign"
    print(f"Model logged at: {best_model_uri}")




**Register Model and Model Versions**

In [0]:
import time
from mlflow.tracking import MlflowClient

client = MlflowClient()
MODEL_NAME = "fashion_mnist_cnn"


best_trial_run_id = study.best_trial.user_attrs.get("mlflow_run_id")

new_acc = client.get_run(best_trial_run_id).data.metrics.get("val_accuracy")
print(f"New model val_accuracy: {new_acc:.4f}")

# Compare vs current staging — only promote if better
promote = False
try:
    current_version = client.get_model_version_by_alias(MODEL_NAME, "staging")
    current_acc = client.get_run(current_version.run_id).data.metrics.get("val_accuracy")
    print(f"Current staging v{current_version.version} val_accuracy: {current_acc:.4f}")
    if new_acc > current_acc:  
        promote = True
        print("New model is better: register and promote")
    else:
        print("New model is NOT better: skip registration")
except Exception:
    promote = True
    print("No current staging model: promote ")

if promote:
    try:
        client.create_registered_model(MODEL_NAME)
    except Exception:
        pass

    mv = client.create_model_version(
        name=MODEL_NAME,
        source=best_model_uri,
        run_id=best_run_id
    )
    for _ in range(10):
        mv = client.get_model_version(MODEL_NAME, mv.version)
        if mv.status == "READY":
            break
        time.sleep(3)

    client.set_registered_model_alias(MODEL_NAME, "staging", mv.version)
    print(f"Registered v{mv.version} | Status: {mv.status}")
    print(f"Alias staging → v{mv.version}")
else:
    print(f"No changes to the staging model → v{current_version.version}")


## **ADD  YOUR ANSWERS TO PART 5 QUESTIONS 1 and 2 HERE**

In [0]:
# Evaluate best model on all test sets
def evaluate_set(name, X, y):
    loss, acc = best_model.evaluate(X, y, verbose=0)
    print(f"{name:<30} accuracy: {acc:.4f}   loss: {loss:.4f}")
    return acc

print("=" * 55)
acc_clean    = evaluate_set("Clean test (no drift)",     X_test_clean,    y_test_clean)
acc_noisy    = evaluate_set("Noisy test (sensor drift)", X_test_noisy,    y_test_clean)
acc_occluded = evaluate_set("Occluded test (blocked)",   X_test_occluded, y_test_clean)
acc_dark     = evaluate_set("Dark test (poor light)",    X_test_dark,     y_test_clean)
print("=" * 55)




**Evaluate Model Performance on Drift Test Sets**

In [0]:
##############PART 6##############

drift_types = ["Noisy", "Occluded\n(blocked)", "Dark\n(poor light)"]
acc_drops   = [(acc_clean - acc_noisy)*100,
               (acc_clean - acc_occluded)*100,
               (acc_clean - acc_dark)*100]

plt.figure(figsize=(6, 4))
plt.bar(drift_types, acc_drops, color=["#e74c3c", "#e67e22", "#8e44ad"])
plt.title("Accuracy Drop by Drift Type")
plt.ylabel("Accuracy Drop (%)")
plt.axhline(y=5, color="black", linestyle="--", label="5% warning threshold")
plt.legend()
plt.tight_layout()
plt.show()


**ADD  YOUR ANSWERS TO PART 6 QUESTION 2 HERE**